# Templisafe - query parameterization use case

## Simple example

In this section we will demonstrate the usage of the main features of the library with a simple example.

### Supported configuration types

You can access all the supported configuration types through the `ContentType` **enum**.

In [1]:
from templisafe import ContentType

ContentType._member_names_, ContentType._member_map_

(['TEXT', 'YAML', 'JSON', 'TOML'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>})

The `TEXT` content type is used for template definitions. Schemas and variants may be defined using any supported configuration language: in this notebook, `YAML` is used.

### `Source` and `SourceSettings` objects

A `Source` represents an abstract input, which can be inline content, a local file or a remote cloud resource. All supported source types are listed in the `SourceKind` **enum**.

In [2]:
from templisafe import SourceKind

SourceKind._member_names_, ContentType._member_map_

(['INLINE', 'LOCAL'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>})

You don’t need to create `Source` objects manually, the library handles them for you. Your only responsibility is to define a `Settings` object for the source, typically a `SourceSettings`, specifying the required configurations.

In the following, we show the different options you have to create a `Settings` object.

#### Creating settings using `SourceSettings.create`

You can easily create a `SourceSettings` using the `SourceSettings.create` method, providing the `SourceKind` and any necessary configuration parameters.

In [3]:
from templisafe import SourceSettings

inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",                  # Enum is automatically parsed 
    content_type="text",            # Enum is automatically parsed 
    content="Hello {{ name }}!", 
)
inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content='Hello {{ name }}!')

#### Creating setting using `SourceSettings.from_<language>`

Alternatively, you can create a `SourceSettings` directly from a configuration string using the `SourceSettings.from_<language>` method.

For example, for a `YAML` configuration you will use the `SourceSettings.from_yaml` method.

In [4]:
json_str: str = '{ "schema": { "var1": "int", "var2": "list[str]" } }'

inline_yaml_settings: str = f"""
kind: inline
content_type: json
content: '{json_str}'
"""

inline_source_settings: SourceSettings = SourceSettings.from_yaml(inline_yaml_settings)
inline_source_settings

InlineSourceSettings(content_type=<ContentType.JSON: 'json'>, content='{ "schema": { "var1": "int", "var2": "list[str]" } }')

Another example: for a `JSON` configuration use `SourceSettings.from_json`.

In [5]:
local_json_settings: str = """
{
    "kind": "local",
    "path": "/tmp/query.sql.j2"
}
"""

local_source_settings: SourceSettings = SourceSettings.from_json(local_json_settings)
local_source_settings

LocalSourceSettings(content_type=None, path='/tmp/query.sql.j2')

### Resources definition

Now that you know how to define `SourceSettings`, let’s proceed to define all the resource configurations required for this use case.

#### Template

Define the **template** using an inline source.

In [6]:
sql_template_content: str = """SELECT 
{%- for col in select_columns %}
  {{ col }}{% if not loop.last %},{% endif %}
{%- endfor %}
FROM users u
  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}
WHERE TRUE 
  AND u.code IN ({{ user_codes | join(', ') }})
  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}
  AND u.status = '{{ user_status }}'
  AND m.code = '{{ metric_code }}'
  AND m.is_updated IS {{ metric_updated_flag }}
  AND m.threshold > {{ metric_threshold }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content=sql_template_content, 
    content_type="text"           # Use text for template contents
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n{%- for col in select_columns %}\n  {{ col }}{% if not loop.last %},{% endif %}\n{%- endfor %}\nFROM users u\n  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}\nWHERE TRUE \n  AND u.code IN ({{ user_codes | join(', ') }})\n  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}\n  AND u.status = '{{ user_status }}'\n  AND m.code = '{{ metric_code }}'\n  AND m.is_updated IS {{ metric_updated_flag }}\n  AND m.threshold > {{ metric_threshold }}\n")

#### Schema

Define the **schema** using an inline source.

In [7]:
schema_content: str = """
schema:
  select_columns:
    type: list
    default:
      - u.id
      - u.name
      - u.code
      - u.age
      - u.status
      - m.is_updated
      - m.threshold
    metadata:
      description: Target select list
      title: SELECT LIST

  user_join_key: 
    type: str
    metadata:
      description: Join key for table user

  metric_join_key: str
    
  user_codes: 
    type: list
    default:
      - 10
      - 11
      - 12
    
  user_age_lower:
    type: int
    default: 0
    constraints:
      ge: 0

  user_age_upper:
    type: int
    default: 1000000
    constraints:
      ge: 0

  user_status: str

  metric_code:
    type: str
    default: "9999910"
    constraints:
      max_length: 12

  metric_updated_flag:
    type: bool
    default: true

  metric_threshold: float
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=schema_content
 )
schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nschema:\n  select_columns:\n    type: list\n    default:\n      - u.id\n      - u.name\n      - u.code\n      - u.age\n      - u.status\n      - m.is_updated\n      - m.threshold\n    metadata:\n      description: Target select list\n      title: SELECT LIST\n\n  user_join_key: \n    type: str\n    metadata:\n      description: Join key for table user\n\n  metric_join_key: str\n\n  user_codes: \n    type: list\n    default:\n      - 10\n      - 11\n      - 12\n\n  user_age_lower:\n    type: int\n    default: 0\n    constraints:\n      ge: 0\n\n  user_age_upper:\n    type: int\n    default: 1000000\n    constraints:\n      ge: 0\n\n  user_status: str\n\n  metric_code:\n    type: str\n    default: "9999910"\n    constraints:\n      max_length: 12\n\n  metric_updated_flag:\n    type: bool\n    default: true\n\n  metric_threshold: float\n')

#### Variants

Define a *single unamed* **variant** using an inline source.

In [8]:
variants_content: str = """
variants:
  select_columns:
    - u.id
    - u.name
    - u.age
    - m.value

  user_join_key: id
  metric_join_key: user_id
  # user_codes -> defaulted
    
  user_age_lower:  18
  user_age_upper: 70
  user_status: ACTIVE

  # metric_code -> defaulted
  metric_updated_flag: true
  metric_threshold: 54.12
"""
variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type="yaml", 
    content=variants_content
)
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nvariants:\n  select_columns:\n    - u.id\n    - u.name\n    - u.age\n    - m.value\n\n  user_join_key: id\n  metric_join_key: user_id\n  # user_codes -> defaulted\n\n  user_age_lower:  18\n  user_age_upper: 70\n  user_status: ACTIVE\n\n  # metric_code -> defaulted\n  metric_updated_flag: true\n  metric_threshold: 54.12\n')

### Compilation

We can now compile the template against the schema to verify that all variables are correctly defined.

#### Create the `Templater`

Create the `Templater` object using default configurations.

In [9]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()

templater: Templater = factory.create()
templater

#### Compile

Compile the template with the given schema.

In [10]:
from templisafe import Compilation

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

<Outcome.SUCCESS: 0>

Inspect the `Compilation` object.

In [11]:
compilation

Compilation(outcome=<Outcome.SUCCESS: 0>, message='Query successfully compiled with schema', diagnostics=(), _spec=CompilationSpec(template=Template(template_str="SELECT \n{%- for col in select_columns %}\n  {{ col }}{% if not loop.last %},{% endif %}\n{%- endfor %}\nFROM users u\n  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}\nWHERE TRUE \n  AND u.code IN ({{ user_codes | join(', ') }})\n  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}\n  AND u.status = '{{ user_status }}'\n  AND m.code = '{{ metric_code }}'\n  AND m.is_updated IS {{ metric_updated_flag }}\n  AND m.threshold > {{ metric_threshold }}\n", vars={'metric_threshold', 'user_age_lower', 'user_status', 'metric_code', 'metric_updated_flag', 'select_columns', 'user_join_key', 'metric_join_key', 'user_age_upper', 'user_codes'}), schema=Schema(model_cls=<class 'abc.ModelSchema'>)))

The schema generated is nothing but a **pydantic model**. 

In [12]:
compilation.compiled.schema.model_cls

abc.ModelSchema

In [13]:
compilation.compiled.schema.model_cls.model_fields

{'select_columns': FieldInfo(annotation=list, required=False, default=['u.id', 'u.name', 'u.code', 'u.age', 'u.status', 'm.is_updated', 'm.threshold'], title='SELECT LIST', description='Target select list', json_schema_extra={'_index': 0}),
 'user_join_key': FieldInfo(annotation=str, required=True, description='Join key for table user', json_schema_extra={'_index': 1}),
 'metric_join_key': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 2}),
 'user_codes': FieldInfo(annotation=list, required=False, default=[10, 11, 12], json_schema_extra={'_index': 3}),
 'user_age_lower': FieldInfo(annotation=int, required=False, default=0, json_schema_extra={'_index': 4}, metadata=[Ge(ge=0)]),
 'user_age_upper': FieldInfo(annotation=int, required=False, default=1000000, json_schema_extra={'_index': 5}, metadata=[Ge(ge=0)]),
 'user_status': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 6}),
 'metric_code': FieldInfo(annotation=str, required=False, default='

#### Compile without a `Schema`

You can also decide to provide no schema when compiling a template: in this case, all variables will be parsed as `object` with default to `None`.

In [14]:
compilation_empty_schema: Compilation = templater.compile(template_source=template_inline_source_settings)      # No schema provided
compilation_empty_schema.outcome

<Outcome.SUCCESS: 0>

In [15]:
compilation_empty_schema.message

'Query successfully compiled with empty schema'

In [16]:
from templisafe import CompilationSpec, Schema

compiled_empty_schema: CompilationSpec = compilation_empty_schema.compiled
empty_schema: Schema = compiled_empty_schema.schema
empty_schema.model_cls.model_fields

{'metric_threshold': FieldInfo(annotation=object, required=False, default=None),
 'user_age_lower': FieldInfo(annotation=object, required=False, default=None),
 'user_status': FieldInfo(annotation=object, required=False, default=None),
 'metric_code': FieldInfo(annotation=object, required=False, default=None),
 'metric_updated_flag': FieldInfo(annotation=object, required=False, default=None),
 'select_columns': FieldInfo(annotation=object, required=False, default=None),
 'user_join_key': FieldInfo(annotation=object, required=False, default=None),
 'metric_join_key': FieldInfo(annotation=object, required=False, default=None),
 'user_age_upper': FieldInfo(annotation=object, required=False, default=None),
 'user_codes': FieldInfo(annotation=object, required=False, default=None)}

### Rendering

After the compilation, we can proceed with the template rendering using the defined **variants**.

In [17]:
from templisafe import InlineSourceSettings

assert isinstance(variants_inline_source_settings, InlineSourceSettings)
print(variants_inline_source_settings.content)


variants:
  select_columns:
    - u.id
    - u.name
    - u.age
    - m.value

  user_join_key: id
  metric_join_key: user_id
  # user_codes -> defaulted

  user_age_lower:  18
  user_age_upper: 70
  user_status: ACTIVE

  # metric_code -> defaulted
  metric_updated_flag: true
  metric_threshold: 54.12



Render the template with the given variants.

In [18]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compilation.compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

<Outcome.SUCCESS: 0>

Inspect the `Rendering` object.

In [19]:
rendering

Rendering(outcome=<Outcome.SUCCESS: 0>, message='Rendering successful', diagnostics=(), _spec=RenderingSpec(_param_by_name={'default_1': Parameterization(variant=Variant(_binding_by_name={'select_columns': Binding(index=0, name='select_columns', value=['u.id', 'u.name', 'u.age', 'm.value']), 'user_join_key': Binding(index=1, name='user_join_key', value='id'), 'metric_join_key': Binding(index=2, name='metric_join_key', value='user_id'), 'user_age_lower': Binding(index=3, name='user_age_lower', value=18), 'user_age_upper': Binding(index=4, name='user_age_upper', value=70), 'user_status': Binding(index=5, name='user_status', value='ACTIVE'), 'metric_updated_flag': Binding(index=6, name='metric_updated_flag', value=True), 'metric_threshold': Binding(index=7, name='metric_threshold', value=54.12)}), rendered_str="SELECT\n  u.id,\n  u.name,\n  u.age,\n  m.value\nFROM users u\n  JOIN metrics m ON u.id = m.user_id\nWHERE TRUE \n  AND u.code IN (10, 11, 12)\n  AND u.age BETWEEN 18 AND 70\n  AND

The variant had no name associated, so a default name was generated.

In [20]:
from templisafe import RenderingSpec

rendered: RenderingSpec = rendering.rendered 
rendered.names

{'default_1'}

Rendered template.

In [21]:
for r in rendered.parameterizations: 
    print(r.rendered_str)

SELECT
  u.id,
  u.name,
  u.age,
  m.value
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (10, 11, 12)
  AND u.age BETWEEN 18 AND 70
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12


Rendered template bindings.

In [22]:
rendered.parameterizations[0].variant.names

{'metric_join_key',
 'metric_threshold',
 'metric_updated_flag',
 'select_columns',
 'user_age_lower',
 'user_age_upper',
 'user_join_key',
 'user_status'}

In [23]:
rendered.parameterizations[0].variant.bindings

[Binding(index=0, name='select_columns', value=['u.id', 'u.name', 'u.age', 'm.value']),
 Binding(index=1, name='user_join_key', value='id'),
 Binding(index=2, name='metric_join_key', value='user_id'),
 Binding(index=3, name='user_age_lower', value=18),
 Binding(index=4, name='user_age_upper', value=70),
 Binding(index=5, name='user_status', value='ACTIVE'),
 Binding(index=6, name='metric_updated_flag', value=True),
 Binding(index=7, name='metric_threshold', value=54.12)]

## Realistic example

In this section we will show a more realistic example, where the resources (**template**, **schema** and **variants**) are defined through configuration files.

### Configuration files

Template definition.

In [24]:
TEMPLATE_PATH: str = "./template.sql.j2"
with open(TEMPLATE_PATH) as f:
    print(f.read())

SELECT 
  {% for col in select_columns -%}
    {{ col }}{% if not loop.last %},{% endif %}
  {% endfor %}
FROM users u
  JOIN metrics m ON u.{{ user_join_key }} = m.{{ metric_join_key }}
WHERE TRUE 
  AND u.code IN (
    {% for code in user_codes -%}
      {{ code }}{% if not loop.last %},{% endif %}
    {% endfor %}
  )
  AND u.age BETWEEN {{ user_age_lower }} AND {{ user_age_upper }}
  AND u.status = '{{ user_status }}'
  AND m.code = '{{ metric_code }}'
  AND m.is_updated IS {{ metric_updated_flag }}
  AND m.threshold > {{ metric_threshold }}


Schema definition.

In [25]:
SCHEMA_PATH: str = "./schema.yaml"
with open(SCHEMA_PATH) as f:
    print(f.read())


schema:
  select_columns:
    type: list
    default:
      - u.id
      - u.name
      - u.code
      - u.age
      - u.status
      - m.is_updated
      - m.threshold

  user_join_key: str

  metric_join_key: str

  user_codes: 
    type: list
    default:
      - 10
      - 11
      - 12
    
  user_age_lower:
    type: int
    default: 0

  user_age_upper:
    type: int
    default: 1000000

  user_status: str

  metric_code:
    type: str
    default: "9999910"

  metric_updated_flag:
    type: bool
    default: true

  metric_threshold: float



Variants definition.

In [26]:
VARIANTS_PATH_1: str = "./variants1.yaml"
with open(VARIANTS_PATH_1) as f:
    print(f.read())

variants:
  - name: adults_only                       # variant name 1
    bindings:
      select_columns:
        - u.id
        - u.name
        - u.age
        - m.value

      user_join_key: id
      metric_join_key: user_id
      # user_codes -> defaulted
        
      user_age_lower:  18
      user_age_upper: 70
      user_status: ACTIVE

      # metric_code -> defaulted
      metric_updated_flag: true
      metric_threshold: 54.12

  - name: minors_only                       # variant name 2
    bindings:
      select_columns:
        - u.id
        - u.name
        - u.age
        - m.value

      user_join_key: id
      metric_join_key: user_id
      # user_codes -> defaulted
        
      user_age_lower:  5
      user_age_upper: 17
      user_status: ACTIVE

      # metric_code -> defaulted
      metric_updated_flag: true
      metric_threshold: 54.12


In [27]:
VARIANTS_PATH_2: str = "./variants2.yaml"
with open(VARIANTS_PATH_2) as f:
    print(f.read())

variants:
  all_ages:                           # variant name
    select_columns:
      - u.id
      - u.name
      - u.age
      - m.value

    user_join_key: id
    metric_join_key: user_id
    # user_codes -> defaulted
      
    # user_age_lower -> defaulted
    # user_age_upper -> defaulted
    user_status: ACTIVE

    # metric_code -> defaulted
    metric_updated_flag: true
    metric_threshold: 54.12


### Sources

Use a `LocalSource` to load configurations from a file. 

When the file uses a standard extension, the content type is inferred automatically.

In [28]:
template_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=TEMPLATE_PATH        # No content_type specified
)

schema_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=SCHEMA_PATH          # No content_type specified
)

variants1_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_1      # No content_type specified
)

variants2_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_2      # No content_type specified
)

template_local_source_settings, schema_local_source_settings, variants1_local_source_settings, variants2_local_source_settings

(LocalSourceSettings(content_type=None, path='./template.sql.j2'),
 LocalSourceSettings(content_type=None, path='./schema.yaml'),
 LocalSourceSettings(content_type=None, path='./variants1.yaml'),
 LocalSourceSettings(content_type=None, path='./variants2.yaml'))

### Build - compilation and rendering in one step

Use `build` to `compile` and `render` in one step.

In [29]:
from templisafe import Build

build: Build = templater.build(
    template_source=template_local_source_settings,
    schema_source=schema_local_source_settings,
    variants_sources=[variants1_local_source_settings, variants2_local_source_settings]
)

build.outcome

<Outcome.SUCCESS: 0>

The `Build` result contains both the `Compilation` and `Rendering` objects.

Inspect the `Compilation`.

In [30]:
compilation: Compilation = build.compilation
compilation.outcome, compilation.message

(<Outcome.SUCCESS: 0>, 'Query successfully compiled with schema')

Inspect the `Rendering`.

In [31]:
rendering: Rendering = build.rendering
rendering.outcome, rendering.message

(<Outcome.SUCCESS: 0>, 'Rendering successful')

In [32]:
rendering.rendered.names

{'adults_only', 'all_ages', 'minors_only'}

In [33]:
from templisafe import Parameterization, Variant

parameterizations: list[Parameterization] = rendering.rendered.parameterizations
for par in parameterizations:
    variant: Variant = par.variant
    print("-" * 50)
    print(f"Variant '{variant.name}':")
    print("-" * 50)
    for b in variant.bindings:
        print(b)

--------------------------------------------------
Variant 'adults_only':
--------------------------------------------------
Binding(index=0, name='select_columns', value=['u.id', 'u.name', 'u.age', 'm.value'])
Binding(index=1, name='user_join_key', value='id')
Binding(index=2, name='metric_join_key', value='user_id')
Binding(index=3, name='user_age_lower', value=18)
Binding(index=4, name='user_age_upper', value=70)
Binding(index=5, name='user_status', value='ACTIVE')
Binding(index=6, name='metric_updated_flag', value=True)
Binding(index=7, name='metric_threshold', value=54.12)
--------------------------------------------------
Variant 'minors_only':
--------------------------------------------------
Binding(index=0, name='select_columns', value=['u.id', 'u.name', 'u.age', 'm.value'])
Binding(index=1, name='user_join_key', value='id')
Binding(index=2, name='metric_join_key', value='user_id')
Binding(index=3, name='user_age_lower', value=5)
Binding(index=4, name='user_age_upper', value=

Note that for the variant '**all_ages**' no bindings for '*user_age_lower*' and '*user_age_upper*' where specified: the variables defaulted to the default value specified in the schema. 

In [34]:
for variant_name, variant in rendering.rendered.mapping.items(): 
    print("-" * 50)
    print(f"Variant '{variant_name}':")
    print("-" * 50)
    print(variant.rendered_str)

--------------------------------------------------
Variant 'adults_only':
--------------------------------------------------
SELECT 
  u.id,
  u.name,
  u.age,
  m.value
  
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (
    10,
    11,
    12
    
  )
  AND u.age BETWEEN 18 AND 70
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12
--------------------------------------------------
Variant 'minors_only':
--------------------------------------------------
SELECT 
  u.id,
  u.name,
  u.age,
  m.value
  
FROM users u
  JOIN metrics m ON u.id = m.user_id
WHERE TRUE 
  AND u.code IN (
    10,
    11,
    12
    
  )
  AND u.age BETWEEN 5 AND 17
  AND u.status = 'ACTIVE'
  AND m.code = '9999910'
  AND m.is_updated IS True
  AND m.threshold > 54.12
--------------------------------------------------
Variant 'all_ages':
--------------------------------------------------
SELECT 
  u.id,
  u.name,
  u.age,
  m.va

<!-- ## Diagnostics examples -->

## Errors validation examples

In this section, we'll show how `templisafe` can safely prevent common template definition errors, both in the **compilation** and **rendering** steps. 

### Diagnostic policy

You can access all the supported configuration types through the `ContentType` **enum**.

In [35]:
from templisafe import ContentType

ContentType._member_names_, ContentType._member_map_

(['TEXT', 'YAML', 'JSON', 'TOML'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>})

When instantiating a `Templater`, you can define its behavior for handling warnings and errors by specifying a `DiagnosticPolicy` from the corresponding enum.

In [36]:
from templisafe import DiagnosticPolicy

DiagnosticPolicy._member_names_, DiagnosticPolicy._member_map_

(['IGNORE', 'LOG', 'STRICT'],
 {'IGNORE': <DiagnosticPolicy.IGNORE: 'ignore'>,
  'LOG': <DiagnosticPolicy.LOG: 'log'>,
  'STRICT': <DiagnosticPolicy.STRICT: 'strict'>})

### Compilation

#### Undeclared variables

An **undeclared variable** is a variable defined in the template but not in the schema, which causes the compilation to fail.

Template

In [37]:
# The variable 'undeclared' is used in the template but will not be defined in the schema
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
  AND m.code = '{{ undeclared }}'     
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content 
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n  AND m.code = '{{ undeclared }}'     \n")

Schema

In [38]:
# No variable 'undeclared' in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=schema_content,
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n')

In [39]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")        # Use ignore policy to avoid raising errors

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

<Outcome.ERROR: 2>

The **compilation** failed. Inspecting the `Compilation` object we can see the error messages.

In [40]:
compilation.message, compilation.diagnostics

('Query compilation failed',
 (Diagnostic(level=<Outcome.ERROR: 2>, message="Undeclared variable: 'undeclared'", name='undeclared', index=None),))

#### Unused variables

An **unused variable** is a variable defined in the schema but not in the template, which causes a warning in the compilation.

Template

In [41]:
# No variable 'unused' in the template
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content
    )
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n")

Schema

In [42]:
# Variable 'unused' defined in the schema but not used in the template
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
  unused: any
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=schema_content,
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n  unused: any\n')

In [43]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="log")       # Use log policy to log warnings and raise errors

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

C:\p\prelios\templisafe\src\templisafe\outcome_handler.py:72: UserWarning: Query compiled with warnings: Unused variable: 'unused'
  warnings.warn(warning_msg, stacklevel=1)


<Outcome.WARNING: 1>

The **compilation** was completed with warnings. Inspecting the `Compilation` object we can see the warning messages.

In [44]:
compilation.message, compilation.diagnostics

('Query successfully compiled with schema',
 (Diagnostic(level=<Outcome.WARNING: 1>, message="Unused variable: 'unused'", name='unused', index=None),))

### Rendering

In the rendering process we start from an already compiled template, consisting in a `Compilation` object.

Template

In [45]:
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content 
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n")

Schema

In [46]:
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",
    content_type="yaml",
    content=schema_content
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n')

In [47]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")        # Use ignore policy to avoid raising errors

compilation: Compilation = templater.compile(
    template_source=template_inline_source_settings,
    schema_source=schema_inline_source_settings
)

compilation.outcome

<Outcome.SUCCESS: 0>

In [48]:
from templisafe import CompilationSpec

compiled: CompilationSpec = compilation.compiled
compiled

CompilationSpec(template=Template(template_str="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n", vars={'col1', 'user_status', 'user_age_lower', 'col2'}), schema=Schema(model_cls=<class 'abc.ModelSchema'>))

In [49]:
compiled.schema.model_cls.model_fields

{'col1': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 0}),
 'col2': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 1}),
 'user_status': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 2}),
 'user_age_lower': FieldInfo(annotation=int, required=True, json_schema_extra={'_index': 3})}

#### Undeclared binding

An **undeclared binding** consists in a variable (without a default) defined in the schema without a corresponding binding in a variant, which causes the rendering to fail.

In [50]:
# The variant is missing the binding 'user_age_lower', 
# which was defined in the schema as a variable without a default
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='variants:\n  col1: id\n  col2: name\n  user_status: ACTIVE\n')

In [51]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

<Outcome.ERROR: 2>

The **rendering** failed. Inspecting the `Rendering` object we can see the error messages.

In [52]:
rendering.message, rendering.diagnostics

('Validation failed due to errors',
 (Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Missing required binding: 'user_age_lower'", name='user_age_lower', index=None),
  Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'user_age_lower': Field required", name='user_age_lower', index=None)))

#### Unused binding

An **unused binding** is a binding defined in a variant without a corresponding variable in the schema, which causes a warning in the rendering.

In [53]:
# Binding 'unused' was not defined in the schema
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
  user_age_lower: 18
  unused: unused 
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='variants:\n  col1: id\n  col2: name\n  user_status: ACTIVE\n  user_age_lower: 18\n  unused: unused \n')

In [54]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

<Outcome.WARNING: 1>

The **rendering** was completed with warnings. Inspecting the `Rendering` object we can see the warning messages.

In [55]:
rendering.message, rendering.diagnostics

('Rendering completed with warnings',
 (Diagnostic(level=<Outcome.WARNING: 1>, message="'default_1' - Extra binding provided: 'unused'", name='unused', index=4),))

#### Wrong typed binding

A **wrong typed binding** is a binding defined in the variant with a *type* different from the one defined in the corresponding schema variable, which causes the rendering to fail.

In [56]:
variants_content: str = """
variants:
  col1: 5.67                      # Should be a string
  col2: name
  user_status: 1                  # Should be a string
  user_age_lower: [1, 2, 3]       # Should be an int
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nvariants:\n  col1: 5.67                      # Should be a string\n  col2: name\n  user_status: 1                  # Should be a string\n  user_age_lower: [1, 2, 3]       # Should be an int\n')

In [57]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants_sources=variants_inline_source_settings
)

rendering.outcome

<Outcome.ERROR: 2>

The **rendering** failed. Inspecting the `Rendering` object we can see the error messages.

In [58]:
rendering.message, rendering.diagnostics

('Validation failed due to errors',
 (Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'col1': Input should be a valid string", name='col1', index=0),
  Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'user_status': Input should be a valid string", name='user_status', index=2),
  Diagnostic(level=<Outcome.ERROR: 2>, message="'default_1' - Invalid value for binding 'user_age_lower': Input should be a valid integer", name='user_age_lower', index=3)))